### NLP Lab 5


In [1]:
import math
from collections import Counter
from itertools import permutations

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test = ["the cat sat on the rug", "the fish sat on the mat"]

## Task 1: Tokenization

**1.1** `tokenize(sentence)`

In [2]:
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]

**1.2** Tokenize all sentences

In [3]:
for s in train + test:
    print(tokenize(s))

['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']


**1.3** Token counts of the training corpus

In [4]:
with_markers = sum(len(tokenize(s)) for s in train)
without_markers = sum(len(s.split()) for s in train)
print("With markers   :", with_markers)
print("Without markers:", without_markers)

With markers   : 30
Without markers: 22


**1.4** `build_vocab` (all words plus `</s>`, no `<s>`)

In [5]:
def build_vocab(sentences):
    vocab = set()
    for s in sentences:
        vocab.update(tokenize(s)[1:])
    return vocab

vocab = build_vocab(train)
V = len(vocab)
print("V =", V)
print(sorted(vocab))

V = 11
['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2: Generating N-grams

**2.1** `get_ngrams(tokens, n)`

In [6]:
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

**2.2** N-grams of the first training sentence

In [7]:
first = tokenize(train[0])
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    print(name, get_ngrams(first, n))

Unigrams [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
Bigrams [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
Trigrams [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]


**2.3** Check that m tokens give m - n + 1 N-grams

In [8]:
for s in train + test:
    m = len(tokenize(s))
    for n in (1, 2, 3):
        assert len(get_ngrams(tokenize(s), n)) == m - n + 1
print("m - n + 1 holds for every sentence and n = 1, 2, 3")

m - n + 1 holds for every sentence and n = 1, 2, 3


**2.4** N larger than the sentence length

In [9]:
print(get_ngrams(["<s>", "hi", "</s>"], 5))

[]


## Task 3: Counting N-grams

**3.1** Build the counts from the training corpus

In [10]:
def count_ngrams(sentences):
    counts = [Counter(), Counter(), Counter()]
    for s in sentences:
        for n, c in zip((1, 2, 3), counts):
            c.update(get_ngrams(tokenize(s), n))
    return counts

uni, bi, tri = count_ngrams(train)

**3.2** Count tables (highest to lowest)

In [11]:
def show(counter):
    for g, c in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
        print(f"  {' '.join(g):<18}{c}")

for name, c in [("Unigram", uni), ("Bigram", bi), ("Trigram", tri)]:
    print(f"{name} counts")
    show(c)

Unigram counts
  the               8
  </s>              4
  <s>               4
  ate               2
  cat               2
  dog               2
  on                2
  sat               2
  bone              1
  fish              1
  mat               1
  rug               1
Bigram counts
  <s> the           4
  ate the           2
  on the            2
  sat on            2
  the cat           2
  the dog           2
  bone </s>         1
  cat ate           1
  cat sat           1
  dog ate           1
  dog sat           1
  fish </s>         1
  mat </s>          1
  rug </s>          1
  the bone          1
  the fish          1
  the mat           1
  the rug           1
Trigram counts
  <s> the cat       2
  <s> the dog       2
  sat on the        2
  ate the bone      1
  ate the fish      1
  cat ate the       1
  cat sat on        1
  dog ate the       1
  dog sat on        1
  on the mat        1
  on the rug        1
  the bone </s>     1
  the cat ate       1
  the cat 

**3.3** Top 5 bigrams and distinct N-gram counts

In [12]:
print("Top 5 bigrams:", bi.most_common(5))
print("Distinct unigrams:", len(uni))
print("Distinct bigrams :", len(bi))
print("Distinct trigrams:", len(tri))

Top 5 bigrams: [(('<s>', 'the'), 4), (('the', 'cat'), 2), (('sat', 'on'), 2), (('on', 'the'), 2), (('ate', 'the'), 2)]
Distinct unigrams: 12
Distinct bigrams : 18
Distinct trigrams: 19


**3.4** `count_of(ngram)`

In [13]:
def count_of(ngram):
    return {1: uni, 2: bi, 3: tri}[len(ngram)].get(ngram, 0)

print(count_of(("the", "cat")))   # seen
print(count_of(("cat", "dog")))   # unseen

2
0


## Task 4: Unigram Model

**4.1** `unigram_prob(w)`, with N = number of predicted tokens (everything except `<s>`)

In [14]:
N = sum(c for (w,), c in uni.items() if w != "<s>")

def unigram_prob(w):
    return count_of((w,)) / N

print("N =", N)

N = 26


**4.2** Probability table and sum check

In [15]:
for w in sorted(vocab):
    print(f"{w:<6}{round(unigram_prob(w), 4)}")
assert math.isclose(sum(unigram_prob(w) for w in vocab), 1)
print("Sum = 1")

</s>  0.1538
ate   0.0769
bone  0.0385
cat   0.0769
dog   0.0769
fish  0.0385
mat   0.0385
on    0.0769
rug   0.0385
sat   0.0769
the   0.3077
Sum = 1


**4.3** Sentence probability

In [16]:
def unigram_sentence_prob(sentence):
    p = 1.0
    for w in tokenize(sentence)[1:]:
        p *= unigram_prob(w)
    return p

for s in test:
    print(f"P({s!r}) = {unigram_sentence_prob(s):.4e}")

P('the cat sat on the rug') = 2.5499e-07
P('the fish sat on the mat') = 1.2749e-07


**4.4** Word order does not matter for a unigram model

In [17]:
words = test[0].split()
orders = sorted(set(permutations(words)))
probs = [unigram_sentence_prob(" ".join(o)) for o in orders]

for o, p in list(zip(orders, probs))[:4]:
    print(" ".join(o), "->", f"{p:.4e}")
print("Distinct orderings:", len(orders))
print("Distinct probabilities:", len({round(p, 15) for p in probs}))

cat on rug sat the the -> 2.5499e-07
cat on rug the sat the -> 2.5499e-07
cat on rug the the sat -> 2.5499e-07
cat on sat rug the the -> 2.5499e-07
Distinct orderings: 360
Distinct probabilities: 1


## Task 5: Bigram Model (MLE)

**5.1** `bigram_prob(prev, w)`

In [18]:
def bigram_prob(prev, w):
    c = count_of((prev,))
    return count_of((prev, w)) / c if c else 0.0

**5.2** `next_word_distribution(prev)`

In [19]:
def next_word_distribution(prev):
    return {w: bigram_prob(prev, w) for w in sorted(vocab) if count_of((prev, w)) > 0}

dist = next_word_distribution("the")
print(dist)
assert math.isclose(sum(dist.values()), 1)
print("Sum = 1")

{'bone': 0.125, 'cat': 0.25, 'dog': 0.25, 'fish': 0.125, 'mat': 0.125, 'rug': 0.125}
Sum = 1


**5.3** Sentence probability with the chain rule

In [20]:
def bigram_sentence_prob(sentence):
    p = 1.0
    for prev, w in get_ngrams(tokenize(sentence), 2):
        p *= bigram_prob(prev, w)
    return p

for s in test:
    print(f"P({s!r}) = {bigram_sentence_prob(s)}")

P('the cat sat on the rug') = 0.015625
P('the fish sat on the mat') = 0.0


**5.4** `predict_next(prev)` (ties broken alphabetically)

In [21]:
def predict_next(prev):
    dist = next_word_distribution(prev)
    return min(dist, key=lambda w: (-dist[w], w)) if dist else None

for w in ["the", "cat", "dog", "sat", "on", "ate"]:
    print(f"{w} -> {predict_next(w)}")

the -> cat
cat -> ate
dog -> ate
sat -> on
on -> the
ate -> the


## Task 6: Trigram Model

**6.1** `trigram_prob(w1, w2, w3)`

In [22]:
def trigram_prob(w1, w2, w3):
    c = count_of((w1, w2))
    return count_of((w1, w2, w3)) / c if c else 0.0

**6.2** Sentence probability (bigram for the first word, trigram for the rest)

In [23]:
def trigram_sentence_prob(sentence):
    t = tokenize(sentence)
    p = bigram_prob(t[0], t[1])
    for i in range(2, len(t)):
        p *= trigram_prob(t[i - 2], t[i - 1], t[i])
    return p

for s in test:
    print(f"P({s!r}) = {trigram_sentence_prob(s)}")

P('the cat sat on the rug') = 0.125
P('the fish sat on the mat') = 0.0


**6.3** Distinct N-grams

In [24]:
print(f"{'N-gram':<10}{'Distinct'}")
for name, c in [("Unigram", uni), ("Bigram", bi), ("Trigram", tri)]:
    print(f"{name:<10}{len(c)}")

N-gram    Distinct
Unigram   12
Bigram    18
Trigram   19


**6.4** Unseen N-grams in the test sentences

In [25]:
for s in test:
    t = tokenize(s)
    ub = [g for g in get_ngrams(t, 2) if count_of(g) == 0]
    ut = [g for g in get_ngrams(t, 3) if count_of(g) == 0]
    print(s)
    print("  unseen bigrams :", ub)
    print("  unseen trigrams:", ut)
    more = "trigram" if len(ut) > len(ub) else "bigram" if len(ub) > len(ut) else "equal"
    print("  more unseen N-grams:", more)

the cat sat on the rug
  unseen bigrams : []
  unseen trigrams: []
  more unseen N-grams: equal
the fish sat on the mat
  unseen bigrams : [('fish', 'sat')]
  unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
  more unseen N-grams: trigram


## Task 7: Zero-Probability Problem

**7.1** `unseen_bigrams(sentence)`

In [26]:
def unseen_bigrams(sentence):
    return [g for g in get_ngrams(tokenize(sentence), 2) if count_of(g) == 0]

**7.2** Unseen bigrams and MLE probability of the test sentences

In [27]:
for s in test:
    print(s, "| unseen:", unseen_bigrams(s), "| P =", bigram_sentence_prob(s))

the cat sat on the rug | unseen: [] | P = 0.015625
the fish sat on the mat | unseen: [('fish', 'sat')] | P = 0.0


**7.3** Word pairs that never occur in training (markers excluded)

In [28]:
words = sorted(vocab - {"</s>"})
pairs = [(a, b) for a in words for b in words]
never = sum(1 for p in pairs if count_of(p) == 0)
print(f"{never} of {len(pairs)} possible pairs never occur = {round(100 * never / len(pairs), 2)}%")

87 of 100 possible pairs never occur = 87.0%


**7.4** `safe_log` and the log probability of the second test sentence

In [29]:
def safe_log(p):
    return math.log(p) if p > 0 else float("-inf")

log_p = sum(safe_log(bigram_prob(a, b)) for a, b in get_ngrams(tokenize(test[1]), 2))
print("ln P(second test sentence) =", log_p)

ln P(second test sentence) = -inf


## Task 8: Smoothing

**8.1** Add-one smoothing

In [30]:
def bigram_prob_addone(prev, w):
    return (count_of((prev, w)) + 1) / (count_of((prev,)) + V)

**8.2** MLE vs add-one

In [31]:
print(f"{'':<16}{'MLE':<10}{'Add-one'}")
for prev, w in [("fish", "sat"), ("the", "cat")]:
    print(f"{f'P({w} | {prev})':<16}{round(bigram_prob(prev, w), 4):<10}{round(bigram_prob_addone(prev, w), 4)}")

                MLE       Add-one
P(sat | fish)   0.0       0.0833
P(cat | the)    0.25      0.1579


**8.3** Smoothed probabilities sum to 1 for every previous word

In [32]:
prevs = [w for (w,) in uni if w != "</s>"]   # </s> never starts a bigram
for prev in prevs:
    assert math.isclose(sum(bigram_prob_addone(prev, w) for w in vocab), 1)
print("Sum = 1 for all", len(prevs), "previous tokens")

Sum = 1 for all 11 previous tokens


**8.4** Add-k smoothing

In [33]:
def bigram_prob_addk(prev, w, k):
    return (count_of((prev, w)) + k) / (count_of((prev,)) + k * V)

print(f"{'k':<8}{'P(sat | fish)'}")
for k in (1, 0.5, 0.1, 0.01):
    print(f"{k:<8}{round(bigram_prob_addk('fish', 'sat', k), 4)}")

k       P(sat | fish)
1       0.0833
0.5     0.0769
0.1     0.0476
0.01    0.009


## Task 9: Perplexity and Model Comparison

**9.1** `perplexity(sentences, prob_fn)`

In [34]:
def perplexity(sentences, prob_fn):
    log_sum, m = 0.0, 0
    for s in sentences:
        for prev, w in get_ngrams(tokenize(s), 2):
            p = prob_fn(prev, w)
            if p == 0:
                return float("inf")
            log_sum += math.log(p)
            m += 1
    return math.exp(-log_sum / m)

**9.2** Model comparison on the test set

In [35]:
def unigram_prob_addone(w):
    return (count_of((w,)) + 1) / (N + V)

results = {
    "Bigram MLE": perplexity(test, bigram_prob),
    "Bigram add-one": perplexity(test, bigram_prob_addone),
    "Unigram add-one": perplexity(test, lambda prev, w: unigram_prob_addone(w)),
}
print(f"{'Model':<18}{'Perplexity'}")
for name, pp in results.items():
    print(f"{name:<18}{round(pp, 4)}")

Model             Perplexity
Bigram MLE        inf
Bigram add-one    5.7969
Unigram add-one   9.137


**9.3** Perplexity of add-k for different k

In [36]:
ks = (1, 0.5, 0.1, 0.01)
test_pp = {k: perplexity(test, lambda p, w: bigram_prob_addk(p, w, k)) for k in ks}

print(f"{'k':<8}{'Perplexity'}")
for k, pp in test_pp.items():
    print(f"{k:<8}{round(pp, 4)}")
print("Best k:", min(test_pp, key=test_pp.get))

k       Perplexity
1       5.7969
0.5     4.5703
0.1     2.9559
0.01    2.6272
Best k: 0.01


**9.4** Choose k on a validation sentence (train on the first 3 sentences, V stays from all 4)

In [37]:
full_counts = (uni, bi, tri)
uni, bi, tri = count_ngrams(train[:3])      # train on 3 sentences only
validation = [train[3]]

val_pp = {k: perplexity(validation, lambda p, w: bigram_prob_addk(p, w, k)) for k in ks}
test_pp3 = {k: perplexity(test, lambda p, w: bigram_prob_addk(p, w, k)) for k in ks}
uni, bi, tri = full_counts                  # restore the full counts

best_val_k = min(val_pp, key=val_pp.get)
best_test_k = min(test_pp3, key=test_pp3.get)

print(f"{'k':<8}{'Validation PP':<16}{'Test PP'}")
for k in ks:
    print(f"{k:<8}{round(val_pp[k], 4):<16}{round(test_pp3[k], 4)}")
print()
print("Best k on validation:", best_val_k, "-> test perplexity", round(test_pp3[best_val_k], 4))
print("Best k on test set  :", best_test_k, "-> test perplexity", round(test_pp3[best_test_k], 4))

k       Validation PP   Test PP
1       8.5857          5.7406
0.5     8.0163          4.4539
0.1     8.026           2.7778
0.01    13.1304         2.4258

Best k on validation: 0.5 -> test perplexity 4.4539
Best k on test set  : 0.01 -> test perplexity 2.4258
